# 04 · CRUD en Neo4j AuraDB

Grafo `(:User)-[:PUSHED|STARRED|FORKED|...]->(:Repo)`. Driver: `neo4j` (Cypher).

**Justificación:** preguntas de conectividad (repositorio con más usuarios, usuarios que comparten repos) son recorridos de grafo.

In [ ]:
import sys, os
sys.path.insert(0, "../src")          # módulos del proyecto (config, extract, clean, ...)
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [ ]:
drv = config.neo4j_driver()
drv.verify_connectivity()
def q(cypher, **p):
    recs = drv.execute_query(cypher, database_=config.NEO4J_DATABASE, **p).records
    return [dict(r) for r in recs]
q('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n')

## C · Crear nodos y relación

In [ ]:
q("MERGE (u:User {login:'estudiante_utec'}) MERGE (r:Repo {name:'utec/bigdata-demo'}) "
  "MERGE (u)-[rel:STARRED]->(r) SET rel.count = 1, rel.first_seen = datetime() RETURN u.login, type(rel), r.name")

## R · Consultar

In [ ]:
q("MATCH (u:User {login:'estudiante_utec'})-[rel]->(r:Repo) RETURN u.login, type(rel), rel.count, r.name")

In [ ]:
# Consulta de grafo real: top 5 repos por usuarios distintos conectados (KPI 3)
q('MATCH (u:User)-[]->(r:Repo) RETURN r.name AS repo, count(DISTINCT u) AS usuarios ORDER BY usuarios DESC LIMIT 5')

In [ ]:
# Usuarios que colaboran en los mismos repositorios que un usuario dado (2 saltos)
top = q('MATCH (u:User)-[]->(:Repo) RETURN u.login AS login, count(*) AS n ORDER BY n DESC LIMIT 1')
if top:
    login = top[0]['login']
    print(login, '->', q('MATCH (a:User {login:$login})-[]->(r:Repo)<-[]-(b:User) WHERE a<>b RETURN b.login AS colega, count(DISTINCT r) AS repos_en_comun ORDER BY repos_en_comun DESC LIMIT 5', login=login))

## U · Actualizar propiedades

In [ ]:
q("MATCH (u:User {login:'estudiante_utec'})-[rel:STARRED]->(r:Repo {name:'utec/bigdata-demo'}) "
  "SET rel.count = rel.count + 1, u.rol = 'estudiante' RETURN u.login, u.rol, rel.count")

## D · Eliminar relación y nodos

In [ ]:
q("MATCH (:User {login:'estudiante_utec'})-[rel:STARRED]->(:Repo {name:'utec/bigdata-demo'}) DELETE rel")
q("MATCH (n) WHERE n.login = 'estudiante_utec' OR n.name = 'utec/bigdata-demo' DETACH DELETE n")
q("MATCH (n) WHERE n.login = 'estudiante_utec' OR n.name = 'utec/bigdata-demo' RETURN count(n) AS quedan")

In [ ]:
drv.close()